In [89]:
import sys
sys.path.insert(0, '/home/frlab/mj_opt/mujoco/legged_robot')

import numpy as np
import mujoco

from core import FloatingBaseRobotState, Pinocchio_Wrapper, Mujoco_Kernel
from utils import SimScheduler, ViewerOverlay
from control.whole_body_controller import WholeBodyController

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [90]:
URDF = '/home/frlab/mj_opt/xmls/systems/g1_description/g1_29dof.urdf'
PKG  = '/home/frlab/mj_opt/xmls/systems/g1_description'
MJCF = '/home/frlab/mj_opt/xmls/systems/g1/scene_29dof.xml'

In [91]:
# ── Pinocchio wrapper 초기화 ──
wrapper = Pinocchio_Wrapper(URDF, PKG)

# ── MuJoCo kernel 초기화 (joint 순서는 pinocchio 기준) ──
joint_names = [wrapper.model.names[i] for i in range(2, wrapper.model.njoints)]
kernel = Mujoco_Kernel(MJCF, joint_names_pin_order=joint_names)

# ── WBC 초기화 ──
wbc = WholeBodyController(wrapper, nc=12, mu=0.5, solver="proxqp")

print(f"nv={wrapper.nv}, na={wrapper.na}, mass={wrapper.mass:.2f} kg")

nv=35, na=29, mass=35.12 kg


In [92]:
# ── CoM 목표 설정 ──
kernel.push_to_wrapper(wrapper)
com_init = wrapper.pos_com_world.copy()
print(f"초기 CoM 위치: {com_init}")

# sinusoidal 파라미터
AMPLITUDE = 0.05    # 진폭 (m)
FREQUENCY = 0.5     # 주파수 (Hz)
OFFSET    = -0.1    # 기준 오프셋 (m), 초기 높이 기준

초기 CoM 위치: [1.95688682e-02 7.21710511e-05 7.21818268e-01]


In [93]:
# ── 콜백 정의 ──
_2pi_f = 2 * np.pi * FREQUENCY

def on_control(t):
    # sinusoidal CoM 목표 (Z축)
    com_des = com_init.copy()
    com_des[2] += OFFSET + AMPLITUDE * np.sin(_2pi_f * t)

    com_dot_des = np.zeros(3)
    com_dot_des[2] = AMPLITUDE * _2pi_f * np.cos(_2pi_f * t)

    # MuJoCo → Pinocchio 동기화
    kernel.push_to_wrapper(wrapper)

    # WBC → torque
    tau = wbc.compute(com_des, com_dot_des)
    kernel.apply_torque_pin(tau)


def on_render(t, overlay):
    com_des = com_init.copy()
    com_des[2] += OFFSET + AMPLITUDE * np.sin(_2pi_f * t)

    overlay.draw_frame_set({
        'L_foot': wrapper.oM_Lfoot,
        'R_foot': wrapper.oM_Rfoot,
        'pelvis': wrapper.oMb,
    }, size=0.12)

    # 현재 CoM (주황)
    overlay.draw_trajectory(wrapper.pos_com_world, radius=0.02,
                            rgba=(1.0, 0.5, 0.0, 1.0))
    # 목표 CoM (초록)
    overlay.draw_trajectory(com_des, radius=0.02,
                            rgba=(0.0, 1.0, 0.0, 0.8))

In [94]:
# ── 시뮬 실행 ──
# ctrl_hz=500, render_hz=60  (viewer 닫으면 종료)
sched = SimScheduler(kernel.model, kernel.data, ctrl_hz=500, render_hz=60)
sched.run(on_control=on_control, on_render=on_render)